In [2]:
from pathlib import Path
import re
import numpy as np
import tifffile as tif
import cv2
from scipy import ndimage as ndi


# ============================================================
# 1. 路径与参数
# ============================================================

input_dir = Path("/home/cyf/wbi/Virginia/registrated_data/f260517/f260517_0530_refined")
output_dir = Path("/home/cyf/wbi/Virginia/registrated_data/f260517/f260517_0530_refined_upsample_regions_ch1")

# 只处理这些帧
frames_to_save = [0, 39, 79, 119, 159, 199]

# 只保存 channel 1
# 注意：
# 如果你的 channel 1 指的是 Python 里的第二个通道，也就是 arr_czyx[1]，这里保持 1。
# 如果你说的 channel 1 指第一个通道，则改成 0。
selected_channel_idx = 1

# XY 上采样倍数
scale_xy = 4

# Z 上采样倍数
# 如果你现在还想保留之前的 Z 方向 2 倍上采样，就保持 2。
# 如果你现在不想 Z 上采样，就改成 1。
scale_z = 2

# TIFF 轴顺序 fallback
fallback_axes = "CZYX"

# 插值方式
xy_interpolation = cv2.INTER_CUBIC
z_order = 1   # 1=linear, 3=cubic；Z 方向建议 linear


# ============================================================
# 2. 两个裁剪区域
#    统一使用 (Z, Y, X) 顺序，因为单通道数据是 stack_zyx
# ============================================================

crop_regions = {
    "region1": (
        slice(0, 16),      # Z: 0:16
        slice(240, 700),   # Y: 240:700
        slice(45, 360),    # X: 45:360
    ),

    "region2": (
        slice(4, 20),  # Z: :-16
        slice(-460,None),  # Y: 800:
        slice(0, 315),   # X: 125:250
    ),
}


# ============================================================
# 3. 工具函数
# ============================================================

def natural_key(path: Path):
    """
    自然排序，避免 frame_10.tif 排在 frame_2.tif 前面。
    """
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", path.name)
    ]


def read_tiff_with_axes(tiff_path: Path, fallback_axes="CZYX"):
    """
    读取 TIFF，并尽量从 tifffile metadata 里获得 axes。
    如果读不到，就使用 fallback_axes。
    """
    with tif.TiffFile(str(tiff_path)) as tf:
        arr = tf.asarray()
        try:
            axes = tf.series[0].axes
        except Exception:
            axes = fallback_axes

    if axes is None or len(axes) != arr.ndim:
        axes = fallback_axes

    return arr, axes


def to_czyx(arr: np.ndarray, axes: str):
    """
    将不同轴顺序统一转成 CZYX: (C, Z, Y, X)
    """
    axes = axes.upper()
    axes = axes.replace("S", "C")

    if set("CZYX").issubset(set(axes)):
        order = [axes.index(a) for a in "CZYX"]
        arr_czyx = np.transpose(arr, order)

    else:
        if arr.ndim != 4:
            raise ValueError(
                f"Cannot infer axes for array with shape {arr.shape}. "
                f"Please set fallback_axes manually."
            )

        shape = arr.shape

        if shape[0] == 2 and shape[1] == 20:
            # (C, Z, Y, X)
            arr_czyx = arr

        elif shape[0] == 20 and shape[1] == 2:
            # (Z, C, Y, X)
            arr_czyx = np.transpose(arr, (1, 0, 2, 3))

        elif shape[-1] == 2 and shape[0] == 20:
            # (Z, Y, X, C)
            arr_czyx = np.transpose(arr, (3, 0, 1, 2))

        else:
            raise ValueError(
                f"Cannot infer axes for shape {shape}. "
                f"Please set fallback_axes manually, e.g. 'CZYX', 'ZCYX', or 'ZYXC'."
            )

    if arr_czyx.shape[0] <= selected_channel_idx:
        raise ValueError(
            f"selected_channel_idx={selected_channel_idx}, "
            f"but input only has {arr_czyx.shape[0]} channels. "
            f"arr_czyx shape={arr_czyx.shape}"
        )

    if arr_czyx.shape[1] != 20:
        print(f"Warning: expected 20 z-slices, but got Z={arr_czyx.shape[1]}")

    return arr_czyx


def validate_crop_region(stack_zyx: np.ndarray, region_name: str, crop_slices):
    """
    检查 crop 后是否为空。
    """
    cropped = stack_zyx[crop_slices]

    if cropped.ndim != 3:
        raise ValueError(
            f"{region_name}: cropped result should be 3D (Z,Y,X), "
            f"but got shape {cropped.shape}"
        )

    if 0 in cropped.shape:
        raise ValueError(
            f"{region_name}: crop result is empty. "
            f"Original shape={stack_zyx.shape}, crop={crop_slices}, cropped shape={cropped.shape}"
        )

    return cropped


def save_upsampled_stack_zyx(
    stack_zyx: np.ndarray,
    save_path: Path,
    scale_xy=4,
    scale_z=2,
    xy_interpolation=cv2.INTER_CUBIC,
    z_order=1,
):
    """
    输入:
        stack_zyx: 单通道 crop 后的 3D 图像，shape = (Z, Y, X)

    输出:
        先对 XY 做 scale_xy 倍上采样，
        再对 Z 做 scale_z 倍上采样，
        最后保存为 TIFF。

    注意:
        这里只处理 crop 后的小 region，不再对整张图做上采样。
    """
    stack_zyx = np.asarray(stack_zyx)

    if stack_zyx.ndim != 3:
        raise ValueError(f"Expected stack_zyx with shape (Z,Y,X), got {stack_zyx.shape}")

    in_dtype = stack_zyx.dtype
    z, h, w = stack_zyx.shape

    # -------------------------
    # Step 1: XY 上采样
    # -------------------------
    up_xy = np.empty((z, h * scale_xy, w * scale_xy), dtype=np.float32)

    for zi in range(z):
        img = stack_zyx[zi].astype(np.float32, copy=False)
        up_xy[zi] = cv2.resize(
            img,
            (w * scale_xy, h * scale_xy),  # cv2: (width, height)
            interpolation=xy_interpolation,
        )

    # -------------------------
    # Step 2: Z 上采样
    # -------------------------
    if scale_z != 1:
        up_zyx_float = ndi.zoom(
            up_xy,
            zoom=(scale_z, 1, 1),
            order=z_order,
        )
    else:
        up_zyx_float = up_xy

    # -------------------------
    # Step 3: 转回原始 dtype
    # -------------------------
    if np.issubdtype(in_dtype, np.integer):
        info = np.iinfo(in_dtype)
        up_zyx = np.clip(up_zyx_float, info.min, info.max)
        up_zyx = np.rint(up_zyx).astype(in_dtype)
    else:
        up_zyx = up_zyx_float.astype(in_dtype, copy=False)

    # -------------------------
    # Step 4: 保存
    # -------------------------
    save_path.parent.mkdir(parents=True, exist_ok=True)

    with tif.TiffWriter(str(save_path), bigtiff=True) as writer:
        for zi in range(up_zyx.shape[0]):
            writer.write(
                up_zyx[zi],
                photometric="minisblack",
                metadata=None,
            )

    return save_path


# ============================================================
# 4. 主处理函数
# ============================================================

def process_folder_regions_ch1(
    input_dir: Path,
    output_dir: Path,
    frames_to_save,
    crop_regions: dict,
    selected_channel_idx=1,
    scale_xy=4,
    scale_z=2,
    fallback_axes="CZYX",
    xy_interpolation=cv2.INTER_CUBIC,
    z_order=1,
):
    """
    对输入文件夹下的 TIFF 按帧处理：

    - 每个 TIFF 对应一帧
    - 每帧包含 2 channels 和 20 z slices
    - 只取 selected_channel_idx
    - 只裁剪 crop_regions 里的两个 region
    - 每个 region 分别保存到 output_dir/region1 和 output_dir/region2
    - 只对 crop 后的小 region 做上采样
    - 不再保存整张图像
    """

    input_files = sorted(
        list(input_dir.glob("*.tif")) + list(input_dir.glob("*.tiff")),
        key=natural_key,
    )

    if len(input_files) == 0:
        raise FileNotFoundError(f"No tif/tiff files found in {input_dir}")

    # 创建 region 输出文件夹
    region_dirs = {}
    for region_name in crop_regions.keys():
        region_dir = output_dir / region_name
        region_dir.mkdir(parents=True, exist_ok=True)
        region_dirs[region_name] = region_dir

    print(f"Found {len(input_files)} TIFF files.")
    print(f"Save only channel index: {selected_channel_idx}")
    print(f"scale_xy={scale_xy}, scale_z={scale_z}")
    print(f"Output dir: {output_dir}")

    for frame_idx in frames_to_save:
        if frame_idx >= len(input_files):
            print(f"[Skip] frame {frame_idx}: only {len(input_files)} files found.")
            continue

        tiff_path = input_files[frame_idx]
        stem = tiff_path.stem

        print(f"\n[Processing] frame {frame_idx}: {tiff_path.name}")

        arr, axes = read_tiff_with_axes(tiff_path, fallback_axes=fallback_axes)
        arr_czyx = to_czyx(arr, axes)

        # arr_czyx: (C, Z, Y, X)
        ch_stack_zyx = arr_czyx[selected_channel_idx]

        print(f"  original channel stack shape: {ch_stack_zyx.shape}")

        for region_name, crop_slices in crop_regions.items():
            region_stack = validate_crop_region(
                ch_stack_zyx,
                region_name=region_name,
                crop_slices=crop_slices,
            )

            print(f"  {region_name}: crop shape {region_stack.shape}")

            save_path = (
                region_dirs[region_name]
                / f"{stem}_frame{frame_idx:04d}_ch{selected_channel_idx}_{region_name}"
                  f"_z{scale_z}x_xy{scale_xy}x.tif"
            )

            save_upsampled_stack_zyx(
                region_stack,
                save_path=save_path,
                scale_xy=scale_xy,
                scale_z=scale_z,
                xy_interpolation=xy_interpolation,
                z_order=z_order,
            )

            print(f"    saved: {save_path}")

    print("\nDone.")


# ============================================================
# 5. 运行
# ============================================================

if __name__ == "__main__":
    process_folder_regions_ch1(
        input_dir=input_dir,
        output_dir=output_dir,
        frames_to_save=frames_to_save,
        crop_regions=crop_regions,
        selected_channel_idx=selected_channel_idx,
        scale_xy=scale_xy,
        scale_z=scale_z,
        fallback_axes=fallback_axes,
        xy_interpolation=xy_interpolation,
        z_order=z_order,
    )

Found 200 TIFF files.
Save only channel index: 1
scale_xy=4, scale_z=2
Output dir: /home/cyf/wbi/Virginia/registrated_data/f260517/f260517_0530_refined_upsample_regions_ch1

[Processing] frame 0: vol_F260517_000000.tif
  original channel stack shape: (20, 1500, 630)
  region1: crop shape (16, 460, 315)
    saved: /home/cyf/wbi/Virginia/registrated_data/f260517/f260517_0530_refined_upsample_regions_ch1/region1/vol_F260517_000000_frame0000_ch1_region1_z2x_xy4x.tif
  region2: crop shape (16, 460, 315)
    saved: /home/cyf/wbi/Virginia/registrated_data/f260517/f260517_0530_refined_upsample_regions_ch1/region2/vol_F260517_000000_frame0000_ch1_region2_z2x_xy4x.tif

[Processing] frame 39: vol_F260517_000039.tif
  original channel stack shape: (20, 1500, 630)
  region1: crop shape (16, 460, 315)
    saved: /home/cyf/wbi/Virginia/registrated_data/f260517/f260517_0530_refined_upsample_regions_ch1/region1/vol_F260517_000039_frame0039_ch1_region1_z2x_xy4x.tif
  region2: crop shape (16, 460, 315)
 

### Sample reference stack

In [18]:
refPath = "/home/cyf/wbi/Virginia/raw_data/f260517/260517_anat_00003_TZCYX.ome.tiff"
movPath ="/home/cyf/wbi/Virginia/raw_data/f260517/260517_exp_00001_TZCYX.ome.tiff"
import sys
import os
import tifffile
import numpy as np
import cupy as cp
from pathlib import Path
import tifffile
import matplotlib.pyplot as plt

sys.path.append('/home/cyf/wbi/Virginia/code_for_paper/wholistic_registration/src/wholistic_registration')
current_dir = os.path.dirname(os.path.abspath("__file__"))
root_dir = os.path.dirname(current_dir)
sys.path.append(root_dir)
from utils import motion_correlation_pattern as mcp
from utils import motion_stage_cache as msc
from utils import IO
ref_stack,_ = IO.readTiff(refPath)
ref_stack_SparseCell = ref_stack[:,0,:,:]

In [19]:
from pathlib import Path
import numpy as np
import tifffile as tif


def sample_stack_at_float_z(
    stack_zyx: np.ndarray,
    z_positions,
    interpolation="linear",
    out_dtype=None,
    z_is_one_based=False,
):
    """
    从 3D reference stack 中，根据浮点 z 坐标采样新的 slice stack。

    参数
    ----
    stack_zyx:
        输入 reference stack，shape = (Z, Y, X)

    z_positions:
        浮点 z 位置列表，例如 [20.7, 30.8, ...]

    interpolation:
        "linear"  : 相邻两个 z slice 线性插值，推荐
        "nearest" : 取最近的整数 z slice

    out_dtype:
        输出 dtype。若为 None，则保持输入 dtype。

    z_is_one_based:
        如果你的 z 坐标来自 MATLAB / ImageJ 的 1-based 编号，设为 True。
        如果是 Python 的 0-based z index，设为 False。

    返回
    ----
    sampled:
        shape = (len(z_positions), Y, X)
    """

    stack_zyx = np.asarray(stack_zyx)

    if stack_zyx.ndim != 3:
        raise ValueError(f"Expected stack shape (Z,Y,X), got {stack_zyx.shape}")

    z_positions = np.asarray(z_positions, dtype=np.float32)

    if z_is_one_based:
        z_positions = z_positions - 1.0

    Z, Y, X = stack_zyx.shape

    if np.any(z_positions < 0) or np.any(z_positions > Z - 1):
        raise ValueError(
            f"Some z_positions are outside valid range [0, {Z - 1}]. "
            f"Got min={z_positions.min()}, max={z_positions.max()}."
        )

    if out_dtype is None:
        out_dtype = stack_zyx.dtype

    if interpolation == "nearest":
        z_idx = np.rint(z_positions).astype(np.int64)
        sampled = stack_zyx[z_idx].astype(np.float32, copy=False)

    elif interpolation == "linear":
        z0 = np.floor(z_positions).astype(np.int64)
        z1 = np.ceil(z_positions).astype(np.int64)

        # 防止刚好落在最后一层时越界
        z1 = np.clip(z1, 0, Z - 1)

        alpha = (z_positions - z0).astype(np.float32)  # shape = (N,)

        sampled = np.empty((len(z_positions), Y, X), dtype=np.float32)

        for i, (a, iz0, iz1) in enumerate(zip(alpha, z0, z1)):
            slice0 = stack_zyx[iz0].astype(np.float32, copy=False)
            slice1 = stack_zyx[iz1].astype(np.float32, copy=False)

            sampled[i] = (1.0 - a) * slice0 + a * slice1

    else:
        raise ValueError("interpolation should be 'linear' or 'nearest'.")

    # 转回目标 dtype
    if np.issubdtype(np.dtype(out_dtype), np.integer):
        info = np.iinfo(out_dtype)
        sampled = np.clip(sampled, info.min, info.max)
        sampled = np.rint(sampled).astype(out_dtype)
    else:
        sampled = sampled.astype(out_dtype, copy=False)

    return sampled

def save_float_z_sampled_tiff(
    ref_stack,
    save_path,
    z_positions,
    interpolation="linear",
    z_is_one_based=False,
):
    """
    读取 reference TIFF，在指定浮点 z 位置采样，并保存为新的 TIFF。
    """

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)


    if ref_stack.ndim != 3:
        raise ValueError(
            f"Expected reference TIFF to be 3D with shape (Z,Y,X), "
            f"but got shape {ref_stack.shape}."
        )

    sampled_stack = sample_stack_at_float_z(
        ref_stack,
        z_positions=z_positions,
        interpolation=interpolation,
        out_dtype=ref_stack.dtype,
        z_is_one_based=z_is_one_based,
    )

    tif.imwrite(
        str(save_path),
        sampled_stack,
        bigtiff=True,
        photometric="minisblack",
        metadata={
            "axes": "ZYX",
            "z_positions": [float(z) for z in z_positions],
            "interpolation": interpolation,
        },
    )

    print(f"Saved sampled stack: {save_path}")
    print(f"Output shape: {sampled_stack.shape}")
    print(f"Output dtype: {sampled_stack.dtype}")

    return sampled_stack

z_positions = [
    20.729996, 30.820724, 40.96902, 49.76937, 60.089676,
    70.075806, 80.4244, 90.17308, 100.47992, 110.924446,
    120.68548, 130.37726, 140.77371, 150.51396, 160.88306,
    170.75919, 180.93715, 190.2979, 200.49612, 210.49219,
]
z_positions = [z + 90 for z in z_positions]
sampled_stack = save_float_z_sampled_tiff(
    ref_stack=ref_stack_SparseCell,
    save_path="/home/cyf/wbi/Virginia/code_for_paper/Fig3/Fig3.1_ref_planes.tiff",
    z_positions=z_positions,
    interpolation="linear",
    z_is_one_based=False,
)


Saved sampled stack: /home/cyf/wbi/Virginia/code_for_paper/Fig3/Fig3.1_ref_planes.tiff
Output shape: (20, 1500, 630)
Output dtype: int16


### Divide moving stacks

In [17]:
from utils import calFlowCrossResolution
mov_stack,_ = IO.readTiff(movPath)
mov_stack_SparseCell = mov_stack[:,:,0,:,:]
timepoints = [0,39,79,119,159,199]
for t in timepoints:
    mov_stack_SparseCell_t = mov_stack_SparseCell[t]
    print(mov_stack_SparseCell_t.shape)
    IO.saveTiff_new(mov_stack_SparseCell_t.reshape(1,20,1,1500,630), f"/home/cyf/wbi/Virginia/code_for_paper/Fig3/Fig3.1/mov_stack_SparseCell_t_{t}.tiff")

(20, 1500, 630)
Saving TIFF file to /home/cyf/wbi/Virginia/code_for_paper/Fig3/Fig3.1/mov_stack_SparseCell_t_0.tiff
  - shape: (1, 20, 1, 1500, 630)
  - spacing: 1, 1
(20, 1500, 630)
Saving TIFF file to /home/cyf/wbi/Virginia/code_for_paper/Fig3/Fig3.1/mov_stack_SparseCell_t_39.tiff
  - shape: (1, 20, 1, 1500, 630)
  - spacing: 1, 1
(20, 1500, 630)
Saving TIFF file to /home/cyf/wbi/Virginia/code_for_paper/Fig3/Fig3.1/mov_stack_SparseCell_t_79.tiff
  - shape: (1, 20, 1, 1500, 630)
  - spacing: 1, 1
(20, 1500, 630)
Saving TIFF file to /home/cyf/wbi/Virginia/code_for_paper/Fig3/Fig3.1/mov_stack_SparseCell_t_119.tiff
  - shape: (1, 20, 1, 1500, 630)
  - spacing: 1, 1
(20, 1500, 630)
Saving TIFF file to /home/cyf/wbi/Virginia/code_for_paper/Fig3/Fig3.1/mov_stack_SparseCell_t_159.tiff
  - shape: (1, 20, 1, 1500, 630)
  - spacing: 1, 1
(20, 1500, 630)
Saving TIFF file to /home/cyf/wbi/Virginia/code_for_paper/Fig3/Fig3.1/mov_stack_SparseCell_t_199.tiff
  - shape: (1, 20, 1, 1500, 630)
  - spac